# Topic Modeling of Policy-Cited Literature

This notebook performs topic modeling on the cleaned publication
datasets produced by the data-preparation workflow.

The analysis includes text preprocessing, document-term matrix
construction, LDA model selection, final topic estimation, topic
interpretation, prevalence analysis, and temporal analysis.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import subprocess
import tempfile

# Project directories
NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "output"

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project directory :", PROJECT_DIR)
print("Data directory    :", DATA_DIR)
print("Output directory  :", OUTPUT_DIR)

Project directory : /nfs/mfirdausi/project/review_paper_2
Data directory    : /nfs/mfirdausi/project/review_paper_2/data
Output directory  : /nfs/mfirdausi/project/review_paper_2/output


## 1. Load Cleaned Publication Data

The cleaned Overton and Scopus publication datasets produced by the
data-preparation notebook are loaded separately. The two sources are
retained as distinct datasets rather than merged.

In [2]:
OVERTON_FILE = (
    DATA_DIR / "overton_full_clean.xlsx"
)

SCOPUS_FILE = (
    DATA_DIR / "scopus_full_clean.xlsx"
)

overton = pd.read_excel(
    OVERTON_FILE
)

scopus = pd.read_excel(
    SCOPUS_FILE
)

print("Overton")
print("-------")
print(f"Documents : {len(overton):,}")
print(f"Columns   : {len(overton.columns):,}")

print("\nScopus")
print("------")
print(f"Documents : {len(scopus):,}")
print(f"Columns   : {len(scopus.columns):,}")

Overton
-------
Documents : 14,266
Columns   : 21

Scopus
------
Documents : 16,403
Columns   : 18


### 1.1 Define the Independent Topic-Modeling Corpora

The Overton and Scopus publication collections are analyzed as
independent corpora. Each corpus therefore receives its own text
preprocessing, document-term matrix, LDA model-selection procedure,
final topic model, and downstream topic analysis.

In [3]:
corpora = {
    "overton": overton.copy(),
    "scopus": scopus.copy(),
}

for corpus_name, corpus_df in corpora.items():

    print(f"{corpus_name.upper()}")
    print("-" * len(corpus_name))

    print(
        f"Documents          : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Non-missing titles : "
        f"{corpus_df['Title'].notna().sum():,}"
    )

    print(
        f"Non-missing abstracts: "
        f"{corpus_df['Abstract'].notna().sum():,}"
    )

    print(
        f"Missing abstracts  : "
        f"{corpus_df['Abstract'].isna().sum():,}"
    )

    print()

OVERTON
-------
Documents          : 14,266
Non-missing titles : 14,266
Non-missing abstracts: 14,264
Missing abstracts  : 2

SCOPUS
------
Documents          : 16,403
Non-missing titles : 16,403
Non-missing abstracts: 16,403
Missing abstracts  : 0



## 2. Prepare Corpora for Topic Modeling

Topic modeling is performed independently for the Overton and Scopus
datasets. Publications without abstracts are excluded because the LDA
models are estimated from abstract text.

In [4]:
lda_corpora = {}

for corpus_name, corpus_df in corpora.items():

    lda_df = (
        corpus_df[
            corpus_df["Abstract"].notna()
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Remove abstracts that are empty after whitespace stripping.
    lda_df["Abstract"] = (
        lda_df["Abstract"]
        .astype(str)
        .str.strip()
    )

    lda_df = (
        lda_df[
            lda_df["Abstract"] != ""
        ]
        .reset_index(drop=True)
    )

    lda_corpora[corpus_name] = lda_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))
    print(
        f"Input publications : "
        f"{len(corpus_df):,}"
    )
    print(
        f"LDA documents      : "
        f"{len(lda_df):,}"
    )
    print(
        f"Excluded           : "
        f"{len(corpus_df) - len(lda_df):,}"
    )
    print()
    

OVERTON
-------
Input publications : 14,266
LDA documents      : 14,264
Excluded           : 2

SCOPUS
------
Input publications : 16,403
LDA documents      : 16,403
Excluded           : 0



### 2.1 Corpus Relevance Diagnostic

Before topic modeling, the retrieved publications are screened
diagnostically for terminology associated with electrical power and
energy systems. This step evaluates whether the search results contain
substantial off-domain literature that could distort the latent topic
structure.

In [5]:
POWER_DOMAIN_TERMS = [
    r"\bpower system",
    r"\bpower systems",
    r"\belectric power",
    r"\belectrical power",
    r"\bpower grid",
    r"\belectric grid",
    r"\belectrical grid",
    r"\bsmart grid",
    r"\bmicrogrid",
    r"\bmicro-grid",
    r"\btransmission system",
    r"\bdistribution system",
    r"\bpower network",
    r"\belectricity",
    r"\bvoltage",
    r"\breactive power",
    r"\bactive power",
    r"\boptimal power flow",
    r"\bload flow",
    r"\bpower flow",
]

power_pattern = re.compile(
    "|".join(POWER_DOMAIN_TERMS),
    flags=re.IGNORECASE,
)

for corpus_name, corpus_df in lda_corpora.items():

    relevant_mask = (
        corpus_df["Title"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
        |
        corpus_df["Abstract"]
        .fillna("")
        .str.contains(
            power_pattern,
            regex=True,
        )
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents             : "
        f"{len(corpus_df):,}"
    )

    print(
        f"Power-domain terminology    : "
        f"{relevant_mask.sum():,}"
    )

    print(
        f"No power-domain terminology : "
        f"{(~relevant_mask).sum():,}"
    )

    print(
        f"Share with domain terminology: "
        f"{relevant_mask.mean() * 100:.2f}%"
    )

    print()

OVERTON
-------
Total documents             : 14,264
Power-domain terminology    : 3,515
No power-domain terminology : 10,749
Share with domain terminology: 24.64%

SCOPUS
------
Total documents             : 16,403
Power-domain terminology    : 10,234
No power-domain terminology : 6,169
Share with domain terminology: 62.39%



In [6]:
relevance_diagnostics = {}

for corpus_name, corpus_df in lda_corpora.items():

    text = (
        corpus_df["Title"].fillna("")
        + " "
        + corpus_df["Abstract"].fillna("")
    )

    relevant_mask = text.str.contains(
        power_pattern,
        regex=True,
    )

    diagnostic_df = (
        corpus_df.loc[
            ~relevant_mask,
            [
                "Title",
                "Year",
                "DOI",
                "Abstract",
            ],
        ]
        .copy()
    )

    relevance_diagnostics[
        corpus_name
    ] = diagnostic_df

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        diagnostic_df[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(
                30,
                len(diagnostic_df),
            ),
            random_state=123,
        )
        .reset_index(drop=True)
    )


OVERTON
-------


,Title,Year,DOI
0,Development and mean life of aluminum first-su...,2009,10.1016/j.solmat.2009.05.004
1,High Reliability Safeguards approach to remote...,2017,10.1016/j.nucengdes.2017.08.012
2,Economic valuation of coccidioidomycosis (vall...,2021,10.1175/WCAS-D-20-0036.1
3,Taxi time prediction at Charlotte airport usin...,2015,10.2514/6.2015-2272
4,Urban Development and Energy Access in Informa...,2016,10.1016/j.proeng.2016.08.680
5,Model order reduction a key technology for dig...,2018,10.1007/978-3-319-75319-5_8
6,Superior room-temperature ductility of typical...,2016,10.1038/ncomms12261
7,Fourier-bessel series model for the Stefan pro...,2018,10.1115/ICONE26-81009
8,A simple apparatus for the injection of lithiu...,2010,10.1016/j.fusengdes.2010.08.033
9,The 'southern model' of welfare in social Europe,1996,10.1177/095892879600600102



SCOPUS
------


,Title,Year,DOI
0,Power optimisation scheme of induction motor u...,2020,10.1049/iet-est.2019.0151
1,An Adaptive Sparse Anisotropic Polynomial-Chao...,2021,10.1109/MEMC.2021.9477248
2,Artificial intelligence in renewable energy te...,2026,10.1016/j.nxener.2026.100575
3,High-performance computing for electric vehicl...,2024,10.4018/978-1-6684-3795-7.ch016
4,Smart energy efficient transportation systems,2026,10.1016/B978-0-323-95045-9.00023-8
5,NEAR-OPTIMAL SOLUTIONS OF CONSTRAINED LEARNING...,2024,0
6,Asynchronous observer-based control for input-...,2026,10.1016/j.cnsns.2025.109458
7,Multi-Objective Optimal Design of an On-Grid H...,2026,10.1109/ACCESS.2026.3667538
8,A review of distributed energy system optimiza...,2023,10.1016/j.jobe.2023.106735
9,Advances in Mountain Gazelle Optimizer: A Comp...,2025,10.1007/s44196-025-00968-4


### 2.2 Domain-Relevance Screening

To support a meaningful comparison between policy-facing and academic
literature, both corpora are restricted to publications relevant to
the power and energy systems domain before topic modeling.

The relevance screen is applied identically to Overton and Scopus.
A broad domain vocabulary is used to capture power-system research
without requiring specific terminology such as "power system" or
"power flow". The screening rule is validated through manual
inspection of retained and excluded records before the filtered
corpora are used for LDA.

In [7]:
# Broad power-and-energy-system terminology used for relevance screening.
#
# This is intentionally broader than the earlier diagnostic.
# The same rule is applied to both Overton and Scopus.

DOMAIN_TERM_GROUPS = {

    "power_system": [
        r"\bpower systems?\b",
        r"\belectric(?:al)? power\b",
        r"\bpower networks?\b",
        r"\belectric(?:al)? networks?\b",
        r"\bpower grids?\b",
        r"\belectric(?:al)? grids?\b",
        r"\bsmart grids?\b",
        r"\bmicrogrids?\b",
        r"\bmicro-grids?\b",
    ],

    "power_flow_operation": [
    r"\bpower flows?\b",
    r"\bload flows?\b",
    r"\boptimal power flows?\b",
    r"\bopf\b",

    # Dispatch and system operation
    r"\beconomic power dispatch\b",
    r"\beconomic dispatch\b",
    r"\boptimal dispatch\b",
    r"\bpower dispatch\b",
    r"\bunit commitment\b",

    # Loss / operating quantities
    r"\bpower loss(?:es)?\b",
    r"\breal power\b",
    r"\bactive power\b",
    r"\breactive power\b",

    # State / voltage / frequency operation
    r"\bstate estimation\b",
    r"\bvoltage stability\b",
    r"\bvoltage control\b",
    r"\bfrequency control\b",
    r"\bfrequency regulation\b",
    ],

    "transmission_distribution": [
        r"\btransmission systems?\b",
        r"\btransmission networks?\b",
        r"\bdistribution systems?\b",
        r"\bdistribution networks?\b",
        r"\bdistribution grids?\b",
        r"\btransmission grids?\b",
        r"\bdistribution feeders?\b",
        r"\bfeeders?\b",
        r"\bsubstations?\b",
    ],

    "electricity": [
        r"\belectricity\b",
        r"\belectric energy\b",
        r"\belectrical energy\b",
        r"\belectricity markets?\b",
        r"\benergy markets?\b",
        r"\belectric utilities?\b",
        r"\bpower utilities?\b",
    ],

    "generation_resources": [
    r"\bpower generation\b",
    r"\belectricity generation\b",
    r"\bgenerating units?\b",
    r"\bgenerators?\b",
    r"\bdistributed generation\b",
    r"\bdistributed energy resources?\b",
    r"\bder\b",
    r"\bders\b",
    r"\benergy resources?\b",
    ],

    "renewables": [
        r"\brenewable energy\b",
        r"\brenewable generation\b",
        r"\bsolar energy\b",
        r"\bsolar power\b",
        r"\bphotovoltaic\b",
        r"\bphotovoltaics\b",
        r"\bpv systems?\b",
        r"\bwind energy\b",
        r"\bwind power\b",
        r"\bwind farms?\b",
        r"\bwind turbines?\b",
    ],

    "storage_ev": [
        r"\benergy storage\b",
        r"\bbattery storage\b",
        r"\bbattery energy storage\b",
        r"\bbess\b",
        r"\belectric vehicles?\b",
        r"\bev charging\b",
        r"\bvehicle-to-grid\b",
        r"\bv2g\b",
    ],

    "power_electronics": [
        r"\bpower electronics\b",
        r"\binverters?\b",
        r"\bconverters?\b",
        r"\bac[- ]dc\b",
        r"\bdc[- ]ac\b",
    ],

    "load_demand": [
        r"\belectric(?:al)? loads?\b",
        r"\bload demand\b",
        r"\belectricity demand\b",
        r"\benergy demand\b",
        r"\bload forecasting\b",
        r"\bdemand response\b",
        r"\bdemand-side management\b",
    ],

    "energy_system": [
    r"\benergy systems?\b",
    r"\bintegrated energy systems?\b",
    r"\bmulti-energy systems?\b",
    r"\bmultienergy systems?\b",
    r"\benergy management systems?\b",
    r"\benergy management\b",
    ],

    "market_reliability": [
    r"\blocational marginal pric(?:e|es|ing)\b",
    r"\blmp\b",
    r"\belectricity pric(?:e|es|ing)\b",
    r"\benergy pric(?:e|es|ing)\b",
    r"\bpower system reliability\b",
    r"\bgrid reliability\b",
    r"\bline failures?\b",
    r"\btransmission line failures?\b",
    r"\bpower system security\b",
    r"\bgrid security\b",
    ],
}

In [8]:
domain_patterns = {
    group: re.compile(
        "|".join(patterns),
        flags=re.IGNORECASE,
    )
    for group, patterns in DOMAIN_TERM_GROUPS.items()
}

domain_screening = {}

for corpus_name, corpus_df in lda_corpora.items():

    screening_df = corpus_df.copy()

    screening_text = (
        screening_df["Title"]
        .fillna("")
        .astype(str)
        + " "
        + screening_df["Abstract"]
        .fillna("")
        .astype(str)
    )

    matched_columns = []

    for group, pattern in domain_patterns.items():

        column = f"match_{group}"

        screening_df[column] = (
            screening_text.str.contains(
                pattern,
                regex=True,
            )
        )

        matched_columns.append(column)

    # Number of different domain concept groups matched.
    screening_df[
        "Domain_Group_Count"
    ] = (
        screening_df[
            matched_columns
        ]
        .sum(axis=1)
    )

    # Initial broad relevance rule:
    # at least one substantive power/energy-domain group.
    screening_df[
        "Domain_Relevant"
    ] = (
        screening_df[
            "Domain_Group_Count"
        ] >= 1
    )

    domain_screening[
        corpus_name
    ] = screening_df

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents       : "
        f"{len(screening_df):,}"
    )

    print(
        f"Domain relevant       : "
        f"{screening_df['Domain_Relevant'].sum():,}"
    )

    print(
        f"Not domain relevant   : "
        f"{(~screening_df['Domain_Relevant']).sum():,}"
    )

    print(
        f"Retention rate        : "
        f"{screening_df['Domain_Relevant'].mean() * 100:.2f}%"
    )

    print()

OVERTON
-------
Total documents       : 14,264
Domain relevant       : 5,045
Not domain relevant   : 9,219
Retention rate        : 35.37%

SCOPUS
------
Total documents       : 16,403
Domain relevant       : 12,042
Not domain relevant   : 4,361
Retention rate        : 73.41%



In [9]:
domain_group_summary = []

for corpus_name, screening_df in domain_screening.items():

    for group in DOMAIN_TERM_GROUPS:

        count = int(
            screening_df[
                f"match_{group}"
            ].sum()
        )

        domain_group_summary.append({
            "Corpus": corpus_name.capitalize(),
            "Domain_Group": group,
            "Documents": count,
            "Percent": (
                count
                / len(screening_df)
                * 100
            ),
        })

domain_group_summary = pd.DataFrame(
    domain_group_summary
)

domain_group_summary.pivot(
    index="Domain_Group",
    columns="Corpus",
    values="Documents",
)

Corpus,Overton,Scopus
Domain_Group,,
electricity,1366,2035
energy_system,564,2390
generation_resources,1282,3485
load_demand,427,1362
market_reliability,177,546
power_electronics,475,1140
power_flow_operation,917,5747
power_system,1989,7107
renewables,2139,4603


In [10]:
for corpus_name, screening_df in domain_screening.items():

    print(f"\n{'=' * 70}")
    print(corpus_name.upper())
    print(f"{'=' * 70}")

    retained = screening_df[
        screening_df["Domain_Relevant"]
    ]

    rejected = screening_df[
        ~screening_df["Domain_Relevant"]
    ]

    print("\nRANDOM RETAINED DOCUMENTS")
    print("-------------------------")

    display(
        retained[
            [
                "Title",
                "Year",
                "DOI",
                "Domain_Group_Count",
            ]
        ]
        .sample(
            n=min(20, len(retained)),
            random_state=123,
        )
        .reset_index(drop=True)
    )

    print("\nRANDOM REJECTED DOCUMENTS")
    print("-------------------------")

    display(
        rejected[
            [
                "Title",
                "Year",
                "DOI",
            ]
        ]
        .sample(
            n=min(20, len(rejected)),
            random_state=123,
        )
        .reset_index(drop=True)
    )


OVERTON

RANDOM RETAINED DOCUMENTS
-------------------------


,Title,Year,DOI,Domain_Group_Count
0,Forecasting day-ahead price of electricity - A...,2013,10.1504/IJBEX.2013.056110,2
1,Exploring electric vehicle charging patterns: ...,2020,10.1016/j.trd.2020.102249,1
2,Condition monitoring of wind turbines: Techniq...,2012,10.1016/j.renene.2012.03.003,1
3,Validation of combined analytical methods to p...,2020,10.1016/j.triboint.2020.106347,1
4,Distributed energy resources and the organized...,2019,10.1016/j.enpol.2018.11.009,3
5,Reduced-order structure-preserving model for p...,2017,10.1109/COMPEL.2017.8013389,2
6,AI-assistance for predictive maintenance of re...,2021,10.1016/j.energy.2021.119775,2
7,Agent-based control framework for distributed ...,2006,10.1109/IAT.2006.27,6
8,Optimization methods applied to renewable and ...,2011,10.1016/j.rser.2010.12.008,2
9,A Setting-Free Differential Protection for Pow...,2019,10.1109/TPWRD.2018.2889471,1



RANDOM REJECTED DOCUMENTS
-------------------------


,Title,Year,DOI
0,The economic burden of physical inactivity: a ...,2016,10.1016/S0140-6736(16)30383-X
1,Challenges to Transforming Unconventional Soci...,2020,10.1017/dmp.2019.92
2,Water transport mechanisms for salt-rejecting ...,2018,10.1016/j.memsci.2018.05.041
3,Associations of mortality with long-term expos...,2015,10.1289/ehp.1408565
4,The impact of synchronisation on secure inform...,2001,10.1007/3-540-45575-2_22
5,Options for reforming agricultural subsidies f...,2022,10.1038/s41467-021-27645-2
6,Distributed snapshots for mobile computing sys...,2004,10.1109/PERCOM.2004.1276856
7,Handling SQL Databases in Automated System Tes...,2020,10.1145/3391533
8,CCured: Type-safe retrofitting of legacy software,2005,10.1145/1065887.1065892
9,Geometry of Kapitsa's potentials,1998,10.1088/0951-7715/11/5/011



SCOPUS

RANDOM RETAINED DOCUMENTS
-------------------------


,Title,Year,DOI,Domain_Group_Count
0,Power systems and microgrids resilience enhanc...,2025,10.1016/j.rser.2024.114953,1
1,Transfer Learning-Based Model Training for Sho...,2026,10.35833/MPCE.2024.000940,3
2,Machine-learned security assessment for changi...,2022,10.1016/j.ijepes.2021.107380,2
3,Robustness of Evolving Power Grids: Modeling a...,2026,10.1201/9781003622130,2
4,Thyristor controlled series compensator planni...,2010,10.1109/PECON.2010.5697550,3
5,Optimal design and operation of a power-to-gas...,2026,10.1016/j.egyr.2026.109680,4
6,Scalable Optimization Methods for Distribution...,2016,10.1109/TSG.2016.2543264,4
7,Probabilistic Power Flow Method for Hybrid AC/...,2023,10.3390/en16062547,4
8,Genetic search for an optimal power flow solut...,2008,0,1
9,Convex Optimization of Power Systems,2015,10.1017/9781139924672,2



RANDOM REJECTED DOCUMENTS
-------------------------


,Title,Year,DOI
0,A Review of Optimal Design for Large-Scale Mic...,2023,10.3390/agronomy13122966
1,Integration of Robust Control and Multi-Object...,2025,10.1002/oca.3249
2,Accuracy analysis of rebar quantity estimation...,2026,10.1080/15623599.2026.2707233
3,Convergence Track Based Adaptive Differential ...,2022,10.32604/cmc.2022.024211
4,Nondestructive detection of trace cadmium in l...,2025,10.1016/j.jfca.2025.108038
5,Survey on Lagrangian relaxation for MILP: impo...,2024,10.1007/s10479-023-05499-9
6,The Comprehensive Review for Biobased FPA Algo...,2021,10.1002/9781119681984.ch7
7,Data-Driven Optimal Scheduling Algorithm of Hu...,2022,10.1155/2022/8602015
8,A modified teaching–learning-based optimizatio...,2019,10.1007/s13042-018-0815-8
9,Dual mutations collaboration mechanism with el...,2022,10.1007/s00500-021-06454-1


### 2.3 Final Domain-Filtered Corpora

In [11]:
lda_corpora_filtered = {}

for corpus_name, screening_df in domain_screening.items():

    filtered_df = (
        screening_df[
            screening_df["Domain_Relevant"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    lda_corpora_filtered[
        corpus_name
    ] = filtered_df

    original_n = len(
        lda_corpora[corpus_name]
    )

    filtered_n = len(
        filtered_df
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Original documents : "
        f"{original_n:,}"
    )

    print(
        f"Retained documents : "
        f"{filtered_n:,}"
    )

    print(
        f"Excluded documents : "
        f"{original_n - filtered_n:,}"
    )

    print(
        f"Retention rate     : "
        f"{filtered_n / original_n * 100:.2f}%"
    )

    print()

OVERTON
-------
Original documents : 14,264
Retained documents : 5,045
Excluded documents : 9,219
Retention rate     : 35.37%

SCOPUS
------
Original documents : 16,403
Retained documents : 12,042
Excluded documents : 4,361
Retention rate     : 73.41%



## 3. R Text-Processing Backend

The abstract corpora are preprocessed using R `tm` and `SnowballC`
through `Rscript`. This preserves the text-processing methodology used
for the LDA analysis while allowing the complete workflow to be
controlled from Python.

In [12]:
from pathlib import Path
import subprocess

RSCRIPT = Path(
    r"/nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript"
)

if not RSCRIPT.exists():
    raise FileNotFoundError(
        f"Rscript not found: {RSCRIPT}"
    )

# Check required R packages.
r_package_check = subprocess.run(
    [
        str(RSCRIPT),
        "-e",
        (
            'pkgs <- c("tm", "SnowballC", "slam", "topicmodels"); '
            'ok <- sapply(pkgs, requireNamespace, quietly=TRUE); '
            'cat(paste(pkgs, ok, sep="="), sep="\\n")'
        ),
    ],
    capture_output=True,
    text=True,
    check=True,
)

print("Rscript:")
print(RSCRIPT)

print("\nRequired R packages:")
print(r_package_check.stdout)

Rscript:
/nfs/mfirdausi/miniconda3/envs/pytorch/bin/Rscript

Required R packages:
tm=TRUE
SnowballC=TRUE
slam=TRUE
topicmodels=TRUE



### 3.1 Text-Preprocessing Configuration

The Overton and Scopus corpora are processed using an identical text
preprocessing configuration to support direct comparison between the
two independently estimated topic models.

Standard English stopwords are supplemented with terms appearing
explicitly in the literature-search query because these terms define
the corpus but provide limited information for distinguishing latent
topics.

In [13]:
# Search-query terms removed from both corpora.
# change this based on your topic

QUERY_STOPWORDS = [
    # Search-query terms
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",

    # Publisher/copyright boilerplate
    "©",
]

# Initial DTM configuration.
MIN_TERM_LENGTH = 3
MIN_DOC_FREQ = 3

print("Query-specific stopwords:")
for word in QUERY_STOPWORDS:
    print(f"  - {word}")

print("\nDTM configuration")
print("-----------------")
print("Minimum term length     :", MIN_TERM_LENGTH)
print("Minimum document freq.  :", MIN_DOC_FREQ)

Query-specific stopwords:
  - power
  - flow
  - machine
  - learning
  - optimization
  - optimisation
  - ©

DTM configuration
-----------------
Minimum term length     : 3
Minimum document freq.  : 3


### 3.2 Preprocess Abstracts with R `tm`

The same R `tm` and `SnowballC` preprocessing pipeline is applied
independently to the Overton and Scopus abstracts. Processing includes
lowercasing, punctuation and number removal, whitespace normalization,
English and query-specific stopword removal, and English Snowball
stemming.

The resulting corpora are used to inspect vocabulary characteristics
before the final document-term matrices are constructed.

In [14]:
R_PREPROCESS_DIR = OUTPUT_DIR / "r_preprocessing"

R_PREPROCESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

R_PREPROCESS_SCRIPT = (
    R_PREPROCESS_DIR / "preprocess_corpus.R"
)

r_preprocess_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file  <- args[1]
output_file <- args[2]

suppressPackageStartupMessages({
    library(tm)
    library(SnowballC)
})

# ------------------------------------------------------------
# Load abstracts
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

abstracts <- data$Abstract

# ------------------------------------------------------------
# Shared stopwords
# ------------------------------------------------------------

custom_stops <- c(
    "power",
    "flow",
    "machine",
    "learning",
    "optimization",
    "optimisation",
    "©"
)

all_stops <- unique(
    c(
        tm::stopwords("english"),
        custom_stops
    )
)

# ------------------------------------------------------------
# tm preprocessing
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(abstracts)
)

corpus <- tm_map(
    corpus,
    content_transformer(tolower)
)

corpus <- tm_map(
    corpus,
    removePunctuation
)

corpus <- tm_map(
    corpus,
    removeNumbers
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    removeWords,
    all_stops
)

# Remove copyright symbol explicitly.
corpus <- tm_map(
    corpus,
    content_transformer(
        function(x) gsub(
            "©",
            " ",
            x,
            fixed = TRUE
        )
    )
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

corpus <- tm_map(
    corpus,
    stemDocument,
    language = "english"
)

corpus <- tm_map(
    corpus,
    stripWhitespace
)

processed <- vapply(
    corpus,
    as.character,
    character(1)
)

result <- data.frame(
    document_id = seq_along(processed),
    processed_text = processed,
    stringsAsFactors = FALSE
)

write.csv(
    result,
    output_file,
    row.names = FALSE,
    fileEncoding = "UTF-8"
)
'''

R_PREPROCESS_SCRIPT.write_text(
    r_preprocess_code,
    encoding="utf-8",
)

print(
    "Created R preprocessing script:",
    R_PREPROCESS_SCRIPT
)

Created R preprocessing script: /nfs/mfirdausi/project/review_paper_2/output/r_preprocessing/preprocess_corpus.R


In [15]:
processed_corpora = {}

for corpus_name, corpus_df in lda_corpora_filtered.items():

    input_file = (
    R_PREPROCESS_DIR
    / f"{corpus_name}_domain_filtered_abstracts.csv"
    )

    output_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )
    # ---------------------------------------------------------
    # Use cached processed corpus if it already exists
    # ---------------------------------------------------------

    if output_file.exists():

        print(
            f"Loading cached {corpus_name.upper()} "
            f"processed corpus ..."
        )

    else:

        print(
            f"Processing {corpus_name.upper()} with R ..."
        )

        # Export abstracts only when preprocessing is required.
        corpus_df[
            ["Abstract"]
        ].to_csv(
            input_file,
            index=False,
            encoding="utf-8",
        )

        run = subprocess.run(
            [
                str(RSCRIPT),
                str(R_PREPROCESS_SCRIPT),
                str(input_file),
                str(output_file),
            ],
            capture_output=True,
            text=True,
            check=True,
        )

    # ---------------------------------------------------------
    # Load processed corpus
    # ---------------------------------------------------------

    processed_df = pd.read_csv(
        output_file,
        keep_default_na=False,
    )

    processed_corpora[
        corpus_name
    ] = processed_df

    print(
        f"Documents processed : "
        f"{len(processed_df):,}"
    )

    print(
        f"Empty documents     : "
        f"{(processed_df['processed_text'].str.strip() == '').sum():,}"
    )

    print()

Loading cached OVERTON processed corpus ...
Documents processed : 5,045
Empty documents     : 0

Loading cached SCOPUS processed corpus ...
Documents processed : 12,042
Empty documents     : 0



In [16]:
for corpus_name, processed_df in processed_corpora.items():

    token_counts = (
        processed_df[
            "processed_text"
        ]
        .str.split()
        .str.len()
    )

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{len(processed_df):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(token_counts.sum()):,}"
    )

    print(
        f"Minimum tokens  : "
        f"{int(token_counts.min()):,}"
    )

    print(
        f"Median tokens   : "
        f"{token_counts.median():.0f}"
    )

    print(
        f"Mean tokens     : "
        f"{token_counts.mean():.1f}"
    )

    print(
        f"Maximum tokens  : "
        f"{int(token_counts.max()):,}"
    )

    print()

OVERTON
-------
Documents       : 5,045
Total tokens    : 564,755
Minimum tokens  : 2
Median tokens   : 109
Mean tokens     : 111.9
Maximum tokens  : 495

SCOPUS
------
Documents       : 12,042
Total tokens    : 1,511,836
Minimum tokens  : 2
Median tokens   : 123
Mean tokens     : 125.5
Maximum tokens  : 452



### 3.3 Inspect Frequent Terms

The most frequent terms remaining after preprocessing are inspected
before constructing the final document-term matrices. This diagnostic
is used to identify high-frequency generic terms that provide little
thematic discrimination and may therefore warrant inclusion in the
shared custom stopword list.

In [17]:
from collections import Counter

term_frequency_tables = {}

for corpus_name, processed_df in processed_corpora.items():

    term_frequency = Counter(
        token
        for text in processed_df["processed_text"]
        for token in text.split()
    )

    top_terms = pd.DataFrame(
        term_frequency.most_common(50),
        columns=[
            "term",
            "frequency",
        ],
    )

    term_frequency_tables[
        corpus_name
    ] = top_terms

    print(f"\n{corpus_name.upper()}")
    print("-" * len(corpus_name))

    display(
        top_terms.head(30)
    )


OVERTON
-------


,term,frequency
0,system,8876
1,energi,8012
2,use,5351
3,model,5189
4,electr,4460
5,distribut,3524
6,paper,3511
7,control,3432
8,wind,3400
9,generat,3343



SCOPUS
------


,term,frequency
0,system,29810
1,energi,20992
2,propos,16506
3,model,15313
4,method,13804
5,distribut,12877
6,use,12276
7,control,12140
8,oper,12072
9,network,11714


## 4. Document-Term Matrix Construction

Separate document-term matrices are constructed for the Overton and
Scopus corpora using R `tm`. The same preprocessing and vocabulary
filtering rules are applied to both corpora.

Terms shorter than three characters and terms occurring in fewer than
three documents are excluded. The resulting matrix dimensions and
sparsity are inspected before LDA model selection.

In [26]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- 3

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)

Created R DTM script: /nfs/mfirdausi/project/review_paper_2/output/r_preprocessing/construct_dtm.R


In [23]:
dtm_summaries = {}
dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    # Domain-filtered processed corpus
    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    # Keep domain-filtered DTM diagnostics separate
    # from the previous unfiltered analysis.
    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    dtm_summaries[
        corpus_name
    ] = summary_df

    dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents       : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Vocabulary      : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens    : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density  : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

OVERTON
-------
Documents       : 5,045
Vocabulary      : 5,840
Nonzero entries : 371,501
Total tokens    : 533,657
Matrix density  : 1.2609%

SCOPUS
------
Documents       : 12,042
Vocabulary      : 8,876
Nonzero entries : 1,034,213
Total tokens    : 1,444,689
Matrix density  : 0.9676%



### 4.1 Vocabulary Filtering

In [24]:
# Compare vocabulary sizes under alternative document-frequency
# thresholds using the document frequencies already calculated in R.

DF_THRESHOLDS = [
    3,
    5,
    10,
    15,
    20,
    25,
    50,
    100,
]

df_threshold_results = []

for corpus_name, terms_df in dtm_term_tables.items():

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        df_threshold_results.append({
            "Corpus": corpus_name.capitalize(),
            "Min_Document_Frequency": threshold,
            "Retained_Terms": int(retained),
        })

df_threshold_results = pd.DataFrame(
    df_threshold_results
)

df_threshold_pivot = (
    df_threshold_results
    .pivot(
        index="Min_Document_Frequency",
        columns="Corpus",
        values="Retained_Terms",
    )
)

df_threshold_pivot

Corpus,Overton,Scopus
Min_Document_Frequency,,
3,5840,8876
5,4323,6374
10,3007,4306
15,2444,3486
20,2096,3048
25,1863,2737
50,1227,1950
100,791,1378


In [25]:
for corpus_name, terms_df in dtm_term_tables.items():

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    n_docs = int(
        dtm_summaries[
            corpus_name
        ].iloc[0]["Documents"]
    )

    for threshold in DF_THRESHOLDS:

        retained = (
            terms_df["Document_frequency"]
            >= threshold
        ).sum()

        print(
            f"DF >= {threshold:3d} "
            f"({threshold / n_docs * 100:5.3f}% docs)"
            f" : {retained:6,d} terms"
        )

    print()

OVERTON
-------
DF >=   3 (0.059% docs) :  5,840 terms
DF >=   5 (0.099% docs) :  4,323 terms
DF >=  10 (0.198% docs) :  3,007 terms
DF >=  15 (0.297% docs) :  2,444 terms
DF >=  20 (0.396% docs) :  2,096 terms
DF >=  25 (0.496% docs) :  1,863 terms
DF >=  50 (0.991% docs) :  1,227 terms
DF >= 100 (1.982% docs) :    791 terms

SCOPUS
------
DF >=   3 (0.025% docs) :  8,876 terms
DF >=   5 (0.042% docs) :  6,374 terms
DF >=  10 (0.083% docs) :  4,306 terms
DF >=  15 (0.125% docs) :  3,486 terms
DF >=  20 (0.166% docs) :  3,048 terms
DF >=  25 (0.208% docs) :  2,737 terms
DF >=  50 (0.415% docs) :  1,950 terms
DF >= 100 (0.830% docs) :  1,378 terms



Based on the threshold analysis, the final vocabulary filter is defined
proportionally rather than using a common absolute document-frequency
count. Terms must occur in at least 0.2% of documents within each
corpus.

This corresponds to a minimum document frequency of 11 documents for
Overton and 25 documents for Scopus. The proportional rule provides
comparable vocabulary filtering despite the different corpus sizes.

### 4.2 Construct the Final Document-Term Matrices

The final document-term matrices are constructed using a minimum term
length of three characters and a minimum document frequency equal to
0.2% of the documents in each corpus.

In [27]:
R_DTM_SCRIPT = (
    R_PREPROCESS_DIR / "construct_dtm.R"
)

r_dtm_code = r'''
args <- commandArgs(trailingOnly = TRUE)

input_file   <- args[1]
summary_file <- args[2]
terms_file   <- args[3]
min_doc_freq <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

# ------------------------------------------------------------
# Load processed documents
# ------------------------------------------------------------

data <- read.csv(
    input_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

# Remove empty documents before DTM construction.
valid <- !is.na(processed) & nzchar(trimws(processed))

processed <- processed[valid]

# ------------------------------------------------------------
# Construct tm corpus
# ------------------------------------------------------------

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Initial DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

# ------------------------------------------------------------
# Minimum document frequency
# ------------------------------------------------------------

doc_freq <- slam::col_sums(
    dtm > 0
)

keep <- doc_freq >= MIN_DOC_FREQ

dtm <- dtm[
    ,
    keep
]

doc_freq <- doc_freq[
    keep
]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

n_docs <- dtm$nrow
n_terms <- dtm$ncol
n_nonzero <- length(dtm$v)
n_tokens <- sum(dtm$v)

density <- (
    n_nonzero
    / (n_docs * n_terms)
    * 100
)

summary_result <- data.frame(
    Documents = n_docs,
    Vocabulary = n_terms,
    Nonzero_entries = n_nonzero,
    Total_tokens = n_tokens,
    Density_percent = density
)

write.csv(
    summary_result,
    summary_file,
    row.names = FALSE
)

term_result <- data.frame(
    Term = Terms(dtm),
    Document_frequency = as.numeric(doc_freq),
    stringsAsFactors = FALSE
)

write.csv(
    term_result,
    terms_file,
    row.names = FALSE
)
'''

R_DTM_SCRIPT.write_text(
    r_dtm_code,
    encoding="utf-8",
)

print(
    "Created R DTM script:",
    R_DTM_SCRIPT
)


Created R DTM script: /nfs/mfirdausi/project/review_paper_2/output/r_preprocessing/construct_dtm.R


In [28]:
import math

MIN_DOC_PERCENT = 0.002

final_dtm_summaries = {}
final_dtm_term_tables = {}

for corpus_name in [
    "overton",
    "scopus",
]:

    processed_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_domain_filtered_processed.csv"
    )

    n_documents = len(
        processed_corpora[corpus_name]
    )

    min_doc_freq = math.ceil(
        n_documents * MIN_DOC_PERCENT
    )

    summary_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_summary.csv"
    )

    terms_file = (
        R_PREPROCESS_DIR
        / f"{corpus_name}_final_dtm_terms.csv"
    )

    subprocess.run(
        [
            str(RSCRIPT),
            str(R_DTM_SCRIPT),
            str(processed_file),
            str(summary_file),
            str(terms_file),
            str(min_doc_freq),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    summary_df = pd.read_csv(
        summary_file
    )

    terms_df = pd.read_csv(
        terms_file
    )

    final_dtm_summaries[
        corpus_name
    ] = summary_df

    final_dtm_term_tables[
        corpus_name
    ] = terms_df

    row = summary_df.iloc[0]

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Documents            : "
        f"{int(row['Documents']):,}"
    )

    print(
        f"Minimum document freq: "
        f"{min_doc_freq:,}"
    )

    print(
        f"DF threshold         : "
        f"{min_doc_freq / n_documents * 100:.3f}%"
    )

    print(
        f"Vocabulary           : "
        f"{int(row['Vocabulary']):,}"
    )

    print(
        f"Nonzero entries      : "
        f"{int(row['Nonzero_entries']):,}"
    )

    print(
        f"Total tokens         : "
        f"{int(row['Total_tokens']):,}"
    )

    print(
        f"Matrix density       : "
        f"{row['Density_percent']:.4f}%"
    )

    print()

OVERTON
-------
Documents            : 5,045
Minimum document freq: 11
DF threshold         : 0.218%
Vocabulary           : 2,865
Nonzero entries      : 356,323
Total tokens         : 512,540
Matrix density       : 2.4652%

SCOPUS
------
Documents            : 12,042
Minimum document freq: 25
DF threshold         : 0.208%
Vocabulary           : 2,737
Nonzero entries      : 988,296
Total tokens         : 1,384,122
Matrix density       : 2.9986%



## 5. LDA Topic Modeling and Model Selection

LDA models are estimated independently for the domain-filtered Overton
and Scopus corpora using R `topicmodels` with Gibbs sampling.

For model selection, each corpus is divided reproducibly into 80%
training and 20% held-out documents using a fixed random seed.
Candidate topic numbers are evaluated using held-out predictive
performance together with topic coherence and distinctiveness.

### 5.1 Training and Held-Out Splits

An independent 80/20 split is generated for each corpus using R's
random-number generator with seed 123. The same sampling procedure is
therefore applied to both datasets.

In [29]:
RANDOM_SEED = 123

lda_splits = {}

for corpus_name, summary_df in final_dtm_summaries.items():

    n_documents = int(
        summary_df.iloc[0]["Documents"]
    )

    split_result = subprocess.run(
        [
            str(RSCRIPT),
            "-e",
            (
                f"set.seed({RANDOM_SEED}); "
                f"n <- {n_documents}; "
                "train_id <- sample("
                "seq_len(n), "
                "size=floor(0.8*n), "
                "replace=FALSE"
                "); "
                'cat(train_id, sep=",")'
            ),
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    # Keep R's 1-based indices because these will
    # subsequently be passed back to R topicmodels.
    train_id = np.fromstring(
        split_result.stdout.strip(),
        sep=",",
        dtype=int,
    )

    all_id = np.arange(
        1,
        n_documents + 1
    )

    test_id = np.setdiff1d(
        all_id,
        train_id
    )

    lda_splits[corpus_name] = {
        "train_id": train_id,
        "test_id": test_id,
    }

    print(corpus_name.upper())
    print("-" * len(corpus_name))

    print(
        f"Total documents    : "
        f"{n_documents:,}"
    )

    print(
        f"Training documents : "
        f"{len(train_id):,}"
    )

    print(
        f"Held-out documents : "
        f"{len(test_id):,}"
    )

    print(
        f"Overlap             : "
        f"{len(np.intersect1d(train_id, test_id))}"
    )

    print()

OVERTON
-------
Total documents    : 5,045
Training documents : 4,036
Held-out documents : 1,009
Overlap             : 0

SCOPUS
------
Total documents    : 12,042
Training documents : 9,633
Held-out documents : 2,409
Overlap             : 0



### 5.2 Coarse Topic-Number Search

A coarse search is performed to identify a plausible range for the
number of latent topics in each domain-filtered corpus. Short Gibbs
chains are used at this screening stage to limit computational cost.

The final corpus-specific vocabulary thresholds established in Section
4 are retained throughout model selection.

In [31]:
K_COARSE = [
    5,
    10,
    15,
    20,
    25,
    30,
    40,
    50,
]

COARSE_BURN_IN = 50
COARSE_ITERATIONS = 100
COARSE_THIN = 10

MIN_DOC_PERCENT = 0.002

print("Coarse K values :", K_COARSE)
print("Burn-in         :", COARSE_BURN_IN)
print("Iterations      :", COARSE_ITERATIONS)
print("Thin            :", COARSE_THIN)

Coarse K values : [5, 10, 15, 20, 25, 30, 40, 50]
Burn-in         : 50
Iterations      : 100
Thin            : 10


In [32]:
R_COARSE_SEARCH_SCRIPT = (
    LDA_DIR / "domain_filtered_coarse_k_search.R"
)

r_coarse_search_code = r'''
args <- commandArgs(trailingOnly = TRUE)

processed_file <- args[1]
train_file     <- args[2]
output_file    <- args[3]
min_doc_freq   <- as.integer(args[4])

suppressPackageStartupMessages({
    library(tm)
    library(slam)
    library(topicmodels)
})

MIN_TERM_LENGTH <- 3
MIN_DOC_FREQ <- min_doc_freq

K_VALUES <- c(
    5, 10, 15, 20,
    25, 30, 40, 50
)

BURN_IN <- 50
ITERATIONS <- 100
THIN <- 10

# ------------------------------------------------------------
# Load domain-filtered processed corpus
# ------------------------------------------------------------

data <- read.csv(
    processed_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    fileEncoding = "UTF-8"
)

processed <- data$processed_text

valid <- (
    !is.na(processed)
    & nzchar(trimws(processed))
)

processed <- processed[valid]

corpus <- VCorpus(
    VectorSource(processed)
)

# ------------------------------------------------------------
# Reconstruct FINAL DTM
# ------------------------------------------------------------

dtm <- DocumentTermMatrix(
    corpus,
    control = list(
        wordLengths = c(
            MIN_TERM_LENGTH,
            Inf
        )
    )
)

doc_freq <- slam::col_sums(
    dtm > 0
)

dtm <- dtm[
    ,
    doc_freq >= MIN_DOC_FREQ
]

# ------------------------------------------------------------
# Train / held-out split
# ------------------------------------------------------------

train_id <- read.csv(
    train_file
)$train_id

test_id <- setdiff(
    seq_len(dtm$nrow),
    train_id
)

dtm_train <- dtm[
    train_id,
]

dtm_test <- dtm[
    test_id,
]

# Retain terms represented in training.
train_terms <- (
    slam::col_sums(
        dtm_train > 0
    ) > 0
)

dtm_train <- dtm_train[
    ,
    train_terms
]

dtm_test <- dtm_test[
    ,
    train_terms
]

# Remove zero-token documents after training-vocabulary filtering.
train_nonempty <- (
    slam::row_sums(dtm_train) > 0
)

test_nonempty <- (
    slam::row_sums(dtm_test) > 0
)

n_empty_train <- sum(!train_nonempty)
n_empty_test <- sum(!test_nonempty)

dtm_train <- dtm_train[
    train_nonempty,
]

dtm_test <- dtm_test[
    test_nonempty,
]

cat(
    "Training documents:",
    dtm_train$nrow,
    "\n"
)

cat(
    "Held-out documents:",
    dtm_test$nrow,
    "\n"
)

cat(
    "Vocabulary:",
    dtm_train$ncol,
    "\n"
)

cat(
    "Minimum document frequency:",
    MIN_DOC_FREQ,
    "\n"
)

cat(
    "Empty training removed:",
    n_empty_train,
    "\n"
)

cat(
    "Empty held-out removed:",
    n_empty_test,
    "\n\n"
)

# ------------------------------------------------------------
# Coarse K search
# ------------------------------------------------------------

results <- data.frame(
    K = integer(),
    Perplexity = numeric(),
    Elapsed_seconds = numeric()
)

for (k in K_VALUES) {

    cat(
        "Fitting K =",
        k,
        "... "
    )

    flush.console()

    start_time <- Sys.time()

    lda_model <- topicmodels::LDA(
        dtm_train,
        k = k,
        method = "Gibbs",
        control = list(
            seed = 123,
            burnin = BURN_IN,
            iter = ITERATIONS,
            thin = THIN
        )
    )

    heldout_perplexity <- (
        topicmodels::perplexity(
            lda_model,
            newdata = dtm_test
        )
    )

    elapsed <- as.numeric(
        difftime(
            Sys.time(),
            start_time,
            units = "secs"
        )
    )

    results <- rbind(
        results,
        data.frame(
            K = k,
            Perplexity = heldout_perplexity,
            Elapsed_seconds = elapsed
        )
    )

    # Preserve completed K values.
    write.csv(
        results,
        output_file,
        row.names = FALSE
    )

    cat(
        sprintf(
            "perplexity = %.4f | %.2f s\n",
            heldout_perplexity,
            elapsed
        )
    )

    flush.console()
}
'''

R_COARSE_SEARCH_SCRIPT.write_text(
    r_coarse_search_code,
    encoding="utf-8",
)

print(
    "Created:",
    R_COARSE_SEARCH_SCRIPT
)

Created: /nfs/mfirdausi/project/review_paper_2/output/lda/domain_filtered_coarse_k_search.R
